In [1]:
!pip install -q pymupdf FlagEmbedding qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.6 MB/s eta 0:00:00


In [2]:
import fitz  # PyMuPDF
import re
import json
import uuid
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional

In [72]:
PDF_DIR = Path("/content")

In [74]:
DOCUMENT_REGISTRY = [
    {
        "filename": "RESOLUCION_3768_DE_2013.pdf",
        "document_type": "resolucion",
        "document_id": "resolucion_3768_2013",
        "document_name": "Resolucion 3768 de 2013 - Ministerio de Transporte",
        "binding": True,
        "fecha": "2013-09-26",
    },
    {
        "filename": "badl7K5f.pdf",
        "document_type": "concepto_juridico",
        "document_id": "concepto_20251340167841",
        "document_name": "Concepto Juridico - Radicado MT 20251340167841 - Rotulado de llantas",
        "binding": False,
        "fecha": "2025-02-17",
    },
    {
        "filename": "NTC__5375.pdf",
        "document_type": "norma_tecnica",
        "document_id": "ntc_5375",
        "document_name": "NTC 5375 - Revision Tecnico-Mecanica y de Emisiones Contaminantes",
        "binding": True,
        "fecha": "2010-10-20",
    },
    {
        "filename": "NORMA_TECNICA_COLOMBIANA-NTC5385.pdf",
        "document_type": "norma_tecnica",
        "document_id": "ntc_5385",  # confirm the actual NTC number once uploaded
        "document_name": "NTC 5385 - Centros de Diagnostico Automotor",
        "binding": True,
        "fecha": None,
    },
]


In [75]:
def extract_text(path: Path) -> str:
    doc = fitz.open(path)
    text = "\n".join(page.get_text() for page in doc)
    doc.close()
    return text


In [76]:
OCR_WORD_FIXES = {
    "SENALIZACION": "SEÑALIZACIÓN",
    "Senalizacion": "Señalización",
    "senalizacion": "señalización",
    "DIRECCION": "DIRECCIÓN",
    "SUSPENSION": "SUSPENSIÓN",
    "TAXIMETRO": "TAXÍMETRO",
    "SISTEME": "SISTEMA",
    "vacio": "vacío",
    "frenometro": "frenómetro",
    "reglones": "renglones",
}


In [77]:
NUMERIC_OCR_PATTERN = re.compile(r"\b(\d)o\b")

In [78]:
def clean_text(text: str) -> str:
    for wrong, right in OCR_WORD_FIXES.items():
        text = text.replace(wrong, right)
    text = NUMERIC_OCR_PATTERN.sub(lambda m: m.group(1) + "0", text)
    # collapse excessive whitespace/newlines from PDF extraction
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


In [79]:
@dataclass
class Chunk:
    text: str
    document_type: str
    document_id: str
    document_name: str
    binding: bool
    fecha: Optional[str] = None
    estado: Optional[str] = None           
    articulo_numero: Optional[int] = None
    modificado_por: Optional[str] = None
    seccion: Optional[str] = None
    severidad: Optional[str] = None
    tipo_vehiculo: Optional[str] = None   
    chunk_id: str = field(default_factory=lambda: str(uuid.uuid4()))


In [80]:
ARTICLE_START = re.compile(r"^ART[IÍ]CULO\s+(\d+)\.\s*(.*)$", re.MULTILINE)
DEROGATED_MARKER = re.compile(r"El texto (?:original|derogado) era el siguiente:?")
MODIFIED_BY = re.compile(r"Modificado por el art\.\s*\d+,\s*Resoluci[oó]n\s*\d+\s*de\s*\d{4}")

In [81]:
NON_LEGAL_MARKERS = [
    "Entendiendo la NTC 5375",
    "Peritaje y avalúo para carros",
    "CDA Movilidad",
]



In [82]:
def strip_non_legal_tail(text: str) -> str:

    cut_points = [text.find(m) for m in NON_LEGAL_MARKERS if text.find(m) != -1]
    if cut_points:
        return text[: min(cut_points)]
    return text



In [83]:

def chunk_resolucion(text: str, meta: dict) -> list[Chunk]:

    text = strip_non_legal_tail(text)
    matches = list(ARTICLE_START.finditer(text))
    chunks = []

    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        block = text[start:end].strip()
        articulo_num = int(m.group(1))

        modif_match = MODIFIED_BY.search(block)
        modificado_por = modif_match.group(0) if modif_match else None

        derog_match = DEROGATED_MARKER.search(block)
        if derog_match:
            vigente_part = block[: derog_match.start()].strip()
            historico_part = block[derog_match.end():].strip()
            chunks.append(Chunk(
                text=vigente_part, articulo_numero=articulo_num,
                estado="vigente", modificado_por=modificado_por, **meta,
            ))
            if historico_part:
                chunks.append(Chunk(
                    text=historico_part, articulo_numero=articulo_num,
                    estado="historico_derogado", modificado_por=modificado_por, **meta,
                ))
        else:
            chunks.append(Chunk(
                text=block, articulo_numero=articulo_num,
                estado="vigente", modificado_por=modificado_por, **meta,
            ))
    return chunks

In [84]:
CONCEPTO_SECTIONS = [
    "CONSULTA", "CONSIDERACIONES", "Marco normativo",
    "Respuesta al interrogante No. 1", "Respuesta al interrogante No. 2",
    "Conclusión",
]

In [85]:

def chunk_concepto(text: str, meta: dict) -> list[Chunk]:
    # Build split points from whichever section headers actually appear
    positions = []
    for header in CONCEPTO_SECTIONS:
        idx = text.find(header)
        if idx != -1:
            positions.append((idx, header))
    positions.sort()

    chunks = []
    for i, (start, header) in enumerate(positions):
        end = positions[i + 1][0] if i + 1 < len(positions) else len(text)
        block = text[start:end].strip()
        if block:
            chunks.append(Chunk(text=block, seccion=header, **meta))
    if not chunks:  
        chunks.append(Chunk(text=text.strip(), **meta))

    return chunks

In [86]:
from collections import defaultdict


In [87]:

SECTION_HEADER = re.compile(
    r"^(\d{1,2}(?:\.\d{1,2})?)\s+([A-ZÁÉÍÓÚÑ][A-ZÁÉÍÓÚÑ \-]{3,60})$",
    re.MULTILINE,
)

In [88]:
def chunk_norma_narrative(text: str, meta: dict) -> list[Chunk]:

    matches = list(SECTION_HEADER.finditer(text))
    chunks = []

    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        block = text[start:end].strip()

        if len(block) > 30:
            chunks.append(Chunk(text=block, seccion=m.group(2).strip(), **meta))

    return chunks

In [89]:
DEFECT_ROW_NOISE = [
    re.compile(r"NORMA T[EÉ]CNICA COLOMBIANA.*?Actualizaci[oó]n\)\s*\d*"),
    re.compile(r"^\d{1,2}(\.\d{1,2}){0,2}\s+[A-ZÁÉÍÓÚÑ][A-ZÁÉÍÓÚÑ \-]{2,60}"),
    re.compile(r"Mediante (?:una?\s+)?inspecci[oó]n sensorial(?:,)?\s+(?:y con ayuda[^,]*,\s*)?se debe (?:detectar|comprobar)[:;]?"),
    re.compile(r"^A\s+B\b"),
]

In [90]:
def strip_defect_row_noise(desc: str) -> str:
    for pattern in DEFECT_ROW_NOISE:
        desc = pattern.sub("", desc)
    return re.sub(r"\s{2,}", " ", desc).strip(" .;:")



In [92]:

def extract_defect_rows_from_pdf(path: Path, meta: dict, col_tolerance: float = 15) -> list[Chunk]:
    """Parse A/B defect tables using word bounding boxes rather than plain text,
    since column position (not the letter itself) is what carries the severity."""
    doc = fitz.open(path)
    chunks = []

    for page in doc:
        words = page.get_text("words")  # x0, y0, x1, y1, word, block, line, word_no
        if not words:
            continue

        # Calibrate this page's column x-positions from the "A"/"B" header cells.
        # Header cells are short isolated tokens "A" and "B" with small bbox width.
        header_candidates = [w for w in words if w[4] in ("A", "B") and (w[2] - w[0]) < 15]
        if len(header_candidates) < 2:
            continue  # no defect table on this page
        col_a_x = min(w[0] for w in header_candidates if w[4] == "A")
        col_b_x = min(w[0] for w in header_candidates if w[4] == "B")

        lines = defaultdict(list)
        for w in words:
            lines[(w[5], w[6])].append(w)

        desc_buffer = []
        for key in sorted(lines.keys()):
            line_words = sorted(lines[key], key=lambda w: w[0])
            x_marks = [w for w in line_words if w[4] == "X"]
            text_words = [w for w in line_words if w[4] != "X"]
            line_text = clean_text(" ".join(w[4] for w in text_words))

            if x_marks:
                xm = x_marks[0]
                if abs(xm[0] - col_a_x) < col_tolerance:
                    sev = "A"
                elif abs(xm[0] - col_b_x) < col_tolerance:
                    sev = "B"
                else:
                    sev = None  # ambiguous - drop rather than guess

                if line_text.strip() and "Descripción del defecto" not in line_text:
                    desc_buffer.append(line_text)
                desc = strip_defect_row_noise(" ".join(desc_buffer).strip())
                desc_buffer = []


                if sev and desc and 10 <= len(desc) <= 400:
                    chunks.append(Chunk(
                        text=f"Defecto: {desc}. Clasificacion: Tipo {sev}.",
                        severidad=sev,
                        **meta,
                    ))
            elif line_text.strip() and "Descripción del defecto" not in line_text:
                desc_buffer.append(line_text)

    doc.close()
    return chunks


In [94]:

def chunk_norma_tecnica(text: str, pdf_path: Path, meta: dict) -> list[Chunk]:

    start_idx = text.find("1. OBJETIVO")
    body = text[start_idx:] if start_idx != -1 else text

    narrative_chunks = chunk_norma_narrative(body, meta)
    defect_chunks = extract_defect_rows_from_pdf(pdf_path, meta)

    return narrative_chunks + defect_chunks


In [95]:
all_chunks: list[Chunk] = []

In [96]:

for doc in DOCUMENT_REGISTRY:
    path = PDF_DIR / doc["filename"]
    if not path.exists():
        print(f"SKIPPING (not found): {doc['filename']}")
        continue

    raw_text = extract_text(path)
    text = clean_text(raw_text)

    meta = {
        "document_type": doc["document_type"],
        "document_id": doc["document_id"],
        "document_name": doc["document_name"],
        "binding": doc["binding"],
        "fecha": doc["fecha"],
    }

    if doc["document_type"] == "resolucion":
        chunks = chunk_resolucion(text, meta)
    elif doc["document_type"] == "concepto_juridico":
        chunks = chunk_concepto(text, meta)
    elif doc["document_type"] == "norma_tecnica":
        chunks = chunk_norma_tecnica(text, path, meta)
    else:
        raise ValueError(f"Unknown document_type: {doc['document_type']}")

    print(f"{doc['document_id']}: {len(chunks)} chunks")
    all_chunks.extend(chunks)


resolucion_3768_2013: 42 chunks
concepto_20251340167841: 6 chunks
ntc_5375: 339 chunks
ntc_5385: 25 chunks


In [97]:
# Quick sanity check - print a few samples per type
for dtype in ("resolucion", "concepto_juridico", "norma_tecnica"):
    sample = next((c for c in all_chunks if c.document_type == dtype), None)
    if sample:
        print(f"\n--- Sample [{dtype}] ---")
        print(json.dumps(asdict(sample), ensure_ascii=False, indent=2)[:600])



--- Sample [resolucion] ---
{
  "text": "ARTÍCULO 1. OBJETO. La presente resolución tiene por objeto establecer las \ncondiciones que deben cumplir los Centros de Diagnóstico Automotor para su \nhabilitación, las líneas móviles para su autorización, funcionamiento, así como \nfijar los criterios y el procedimiento para realizar las revisiones técnico-mecánicas \ny de emisiones contaminantes de los vehículos automotores que transiten por el \nterritorio nacional.",
  "document_type": "resolucion",
  "document_id": "resolucion_3768_2013",
  "document_name": "Resolucion 3768 de 2013 - Ministerio de Transporte",
  "binding":

--- Sample [concepto_juridico] ---
{
  "text": "CONSULTA\n“Respecto de los requisitos denominados: “la interpretación de la nomenclatura e índices\nde rotulado” y “las indicaciones de instalación”, surgen los siguientes interrogantes, para\nel caso del Folleto de Usuario, para el caso de comercialización de llantas: \na) ¿Los “índices de rotulado”, son lo mismo que, 

In [98]:
from FlagEmbedding import BGEM3FlagModel

In [99]:
model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [100]:
texts = [c.text for c in all_chunks]

In [101]:
embeddings = model.encode(texts, batch_size=12, max_length=512)["dense_vecs"]




pre tokenize:   0%|          | 0/35 [00:00<?, ?it/s]

pre tokenize:  11%|█▏        | 4/35 [00:00<00:00, 31.05it/s]

pre tokenize:  23%|██▎       | 8/35 [00:00<00:00, 28.49it/s]

pre tokenize:  71%|███████▏  | 25/35 [00:00<00:00, 80.33it/s]

pre tokenize: 100%|██████████| 35/35 [00:00<00:00, 69.22it/s]


Inference Embeddings:   0%|          | 0/35 [00:00<?, ?it/s]

Inference Embeddings:   3%|▎         | 1/35 [01:16<43:20, 76.49s/it]

Inference Embeddings:   6%|▌         | 2/35 [02:33<42:23, 77.07s/it]

Inference Embeddings:   9%|▊         | 3/35 [03:35<37:24, 70.14s/it]

Inference Embeddings:  11%|█▏        | 4/35 [04:28<32:43, 63.34s/it]

Inference Embeddings:  14%|█▍        | 5/35 [05:03<26:26, 52.89s/it]

Inference Embeddings:  17%|█▋        | 6/35 [05:26<20:41, 42.81s/it]

Inference Embeddings:  20%|██        | 7/35 [05:45<16:21, 35.05s/it]

Inference Embeddings:  23%|██▎       | 8/35 [06:08<14:06, 31.37s/it]

Inference Embeddings:  26%|██▌       | 9/35 [06:24<11:23, 26.28s/it]

I

In [102]:
print(f"Generated {len(embeddings)} embeddings of dim {embeddings.shape[1]}")


Generated 412 embeddings of dim 1024


In [103]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

In [104]:
QDRANT_URL = "QDRANT-CLUSTER-URL"
QDRANT_API_KEY = "API-KEY"
COLLECTION_NAME = "COLLECTION_NAME"


In [105]:
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)


In [123]:
if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
    )
    print(f"Created collection: {COLLECTION_NAME}")
else:
    print(f"Collection already exists: {COLLECTION_NAME}")

client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="document_id",
    field_schema="keyword",
)
print(f"Indexed 'document_id' for collection: {COLLECTION_NAME}")

Collection already exists: compliance_normativa
Indexed 'document_id' for collection: compliance_normativa


In [107]:
points = [
    PointStruct(
        id=chunk.chunk_id,
        vector=embeddings[i].tolist(),
        payload=asdict(chunk),
    )
    for i, chunk in enumerate(all_chunks)
]

In [108]:
BATCH = 64

In [109]:
for i in range(0, len(points), BATCH):
    client.upsert(collection_name=COLLECTION_NAME, points=points[i : i + BATCH])

print(f"Uploaded {len(points)} points to '{COLLECTION_NAME}'")


Uploaded 412 points to 'compliance_normativa'


In [110]:

def search(query: str, top_k: int = 5, filter_binding: Optional[bool] = None):
    query_vec = model.encode([query])["dense_vecs"][0].tolist()
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vec,
        limit=top_k,
    ).points
    for r in results:
        p = r.payload
        tag = "[NO VINCULANTE]" if p["binding"] is False else "[VINCULANTE]"
        print(f"\nscore={r.score:.3f} {tag} {p['document_id']} "
              f"art.{p.get('articulo_numero')} sev.{p.get('severidad')}")
        print(p["text"][:250])


In [111]:
# Example test queries - adjust to what Car Inspector will actually ask
search("requisitos para habilitar un Centro de Diagnostico Automotor")
search("defecto sistema de frenos tipo A")
search("indice de rotulado de llantas es obligatorio en el folleto de usuario")


score=0.728 [VINCULANTE] resolucion_3768_2013 art.6 sev.None
ARTÍCULO 6. REQUISITOS DE HABILITACIÓN. Los Centros de Diagnóstico 
Automotor interesados en la prestación del servicio de revisión técnico-mecánica 
y de emisiones contaminantes deben solicitar habilitación ante la Subdirección de 
Tránsito del Mini

score=0.719 [VINCULANTE] resolucion_3768_2013 art.11 sev.None
ARTÍCULO 11. OBLIGACIONES DE LOS CENTROS DE DIAGNÓSTICO 
AUTOMOTOR. Una vez habilitado el Centro de Diagnóstico Automotor para 
operar en la sede solicitada, este deberá: 
 
a) Modificado por el art. 4, Resolución 6589 de 2019. <El nuevo texto es el 

score=0.712 [VINCULANTE] resolucion_3768_2013 art.4 sev.None
ARTÍCULO 4. HABILITACIÓN DE LOS CENTROS DE DIAGNÓSTICO 
AUTOMOTOR. Los Centros de Diagnóstico Automotor interesados en la 
prestación 
del 
servicio 
de 
revisión 
técnico-mecánica 
y de emisiones 
contaminantes deberán obtener habilitación por parte

score=0.697 [VINCULANTE] resolucion_3768_2013 art.1 sev.Non

In [112]:

import fitz

In [116]:


path = "/content/NTC__5375.pdf" 
doc = fitz.open(path)
full_text = "\n".join(page.get_text() for page in doc)

print(f"Paginas: {len(doc)}")
print(f"Caracteres totales: {len(full_text)}")
print(f"Contiene '1. OBJETIVO': {'1. OBJETIVO' in full_text}")
print(f"Contiene 'OBJETO': {'OBJETO' in full_text}")

ab_header_pages = 0
for page in doc:
    words = page.get_text("words")
    ab = [w for w in words if w[4] in ("A", "B") and (w[2]-w[0]) < 15]
    if len(ab) >= 2:
        ab_header_pages += 1
print(f"Paginas con posible tabla A/B: {ab_header_pages}")

print("\n--- Primeros 500 caracteres ---")
print(full_text[:500])
print("\n--- Ultimos 500 caracteres ---")
print(full_text[-500:])
doc.close()




Paginas: 43
Caracteres totales: 83521
Contiene '1. OBJETIVO': True
Contiene 'OBJETO': False
Paginas con posible tabla A/B: 35

--- Primeros 500 caracteres ---
NORMA TÉCNICA  
 
  
    NTC  
COLOMBIANA 
 
 
 
   5375 
 
 
2010-10-20 
 
 
REVISION TECNICO- MECANICA Y DE EMISIONES  
CONTAMINANTES EN VEHICULOS AUTOMOTORES 
 
 
 
 
 
 
E: 
TECHNICAL-MECHANICAL AND POLUTION EMISSION 
INSPECTIONS IN AUTOMOTIVE VEHICLES  
 
CoRRESPoNDENCIA: 
  
DESCRIPTORES:  
vehículos automotores – revisión 
técnico 
mecánica; 
vehículos 
automotores – revisión emisiones 
contaminantes. 
 
 
 
 
I.C.S.: 43.18o.oo 
 
  
Editada por el Instituto Colombiano de Normas Técnicas 

--- Ultimos 500 caracteres ---
 
 
La ubicación de la placa en el techo en lugar diferente el eje longitudinal del vehículo 
cualquiera sea la clase del vehículo (debe estar colocada en cualquier punto a lo largo 
del eje longitudinal en forma perpendicular y centrada transversalmente). 
 
 
La ubicación de la placa en la parte externa l

In [124]:

from qdrant_client.models import Filter, FieldCondition, MatchValue

count_before = client.count(
    collection_name=COLLECTION_NAME,
    count_filter=Filter(
        must=[FieldCondition(key="document_id", match=MatchValue(value="ntc_5385"))]
    ),
).count
print(f"Puntos con document_id='ntc_5385' antes de borrar: {count_before}")


client.delete(
    collection_name=COLLECTION_NAME,
    points_selector=Filter(
        must=[FieldCondition(key="document_id", match=MatchValue(value="ntc_5385"))]
    ),
)


count_after = client.count(
    collection_name=COLLECTION_NAME,
    count_filter=Filter(
        must=[FieldCondition(key="document_id", match=MatchValue(value="ntc_5385"))]
    ),
).count
total = client.count(collection_name=COLLECTION_NAME).count
print(f"Puntos con document_id='ntc_5385' despues de borrar: {count_after}")
print(f"Total de puntos en la coleccion: {total}")


Puntos con document_id='ntc_5385' antes de borrar: 25
Puntos con document_id='ntc_5385' despues de borrar: 0
Total de puntos en la coleccion: 387


In [125]:
search("Estado de Farolas o lamparas en vehiculos de cuatro ruedas")



score=0.555 [VINCULANTE] ntc_5375 art.None sev.A
Defecto: Mal estado o el no funcionamiento de la(s) luz (luces) de parada o freno. Clasificacion: Tipo A.

score=0.539 [VINCULANTE] ntc_5375 art.None sev.None
6.4 ALUMBRADO Y SEÑALIZACIÓN 
 
6.4.1 Mediante inspección sensorial se debe detectar: 
 
Descripción del defecto 
A 
B 
El no funcionamiento de los comandos que encienden y conmutan las luces. 
X 
 
Mal estado (con riesgo de desprendimiento o ausenci

score=0.528 [VINCULANTE] ntc_5375 art.None sev.A
Defecto: Mal estado (con riesgo de desprendimiento o ausencia de las pastas o vidrios) o el no funcionamiento del sistema o conjunto de luces de parada y freno. Clasificacion: Tipo A.

score=0.528 [VINCULANTE] ntc_5375 art.None sev.A
Defecto: Mal estado (con riesgo de desprendimiento o ausencia de las pastas o vidrios) o el no funcionamiento del sistema o conjunto de luces de parada y freno. Clasificacion: Tipo A.

score=0.528 [VINCULANTE] ntc_5375 art.None sev.A
Defecto: Mal estado (c

In [126]:
# Diagnostico: cuenta duplicados exactos por texto
total = client.count(collection_name=COLLECTION_NAME).count
print(f"Total de puntos actuales en la coleccion: {total}")
print(f"Esperado si no hay duplicados: 387")

if total != 387:
    print("\n--- Hay duplicados. Recreando la coleccion desde cero. ---")

    # Borra y recrea la coleccion vacia
    client.delete_collection(COLLECTION_NAME)
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
    )
    print(f"Coleccion '{COLLECTION_NAME}' recreada vacia.")


    points = [
        PointStruct(id=chunk.chunk_id, vector=embeddings[i].tolist(), payload=asdict(chunk))
        for i, chunk in enumerate(all_chunks)
    ]
    BATCH = 64
    for i in range(0, len(points), BATCH):
        client.upsert(collection_name=COLLECTION_NAME, points=points[i:i+BATCH])

    total_final = client.count(collection_name=COLLECTION_NAME).count
    print(f"Total final tras recarga limpia: {total_final}")
else:
    print("Todo en orden, no hay duplicados.")



Total de puntos actuales en la coleccion: 387
Esperado si no hay duplicados: 387
Todo en orden, no hay duplicados.
